In [8]:
# Cell 1: 环境
import os, sys, json
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
from src.ci.patent_service import create_patent_family, list_patent_families_for_organization
from src.ci.clinical_trial_service import create_clinical_trial, list_trials_for_program
driver = get_driver()
print("✅ 模块导入成功")

✅ 模块导入成功


In [15]:
# Cell A: 给 TechnicalClaim / TechnicalResult 补 SourceArtifact
from src.shared.source_artifact_service import create_source_artifact

with driver.session() as s:
    demo_src_id = create_source_artifact(
        driver,
        source_type="literature",
        title="Stage2 demo evidence source",
        url="https://example.com/stage2-demo-source",
        published_date="2026-01-01",
        credibility_tier="secondary",
        actor_id="fix",
    )
    print(f"✅ 通用 demo source: {demo_src_id}")

    claims = list(s.run("""
        MATCH (c:TechnicalClaim)
        WHERE NOT EXISTS { MATCH (c)-[:SUPPORTED_BY]->(:SourceArtifact) }
        RETURN c.claim_id AS cid
    """))
    for c in claims:
        s.run("""
            MATCH (c:TechnicalClaim {claim_id: $cid})
            MATCH (src:SourceArtifact {source_id: $sid})
            MERGE (c)-[:SUPPORTED_BY]->(src)
        """, cid=c["cid"], sid=demo_src_id)
    print(f"✅ 补齐 {len(claims)} 个 Claim 来源")

    results = list(s.run("""
        MATCH (r:TechnicalResult)
        WHERE NOT EXISTS { MATCH (r)-[:REPORTED_IN]->(:SourceArtifact) }
        RETURN r.result_id AS rid
    """))
    for r in results:
        s.run("""
            MATCH (r:TechnicalResult {result_id: $rid})
            MATCH (src:SourceArtifact {source_id: $sid})
            MERGE (r)-[:REPORTED_IN]->(src)
        """, rid=r["rid"], sid=demo_src_id)
    print(f"✅ 补齐 {len(results)} 个 Result 来源")

✅ 通用 demo source: SRC:6DB77F74
✅ 补齐 10 个 Claim 来源
✅ 补齐 10 个 Result 来源


In [16]:
# Cell B: 给 TechnicalClaim 批量审核
from src.shared.review import create_review

with driver.session() as s:
    claims = list(s.run("""
        MATCH (c:TechnicalClaim)
        WHERE NOT EXISTS { MATCH (:Review)-[:REVIEWS]->(c) }
        RETURN c.claim_id AS cid
    """))

ok = 0
for c in claims:
    try:
        create_review(
            driver,
            review_type="technical_intelligence_review",
            target_object_type="TechnicalClaim",
            target_object_id=c["cid"],
            reviewer_id="expert_wang",
            decision="approved",
            comment="技术主张审核通过",
            update_target_status=True,
            actor_id="fix",
        )
        ok += 1
    except Exception as e:
        print(f"  ❌ {c['cid']}: {e}")
print(f"✅ 审核 {ok}/{len(claims)} 个 TechnicalClaim")

✅ 审核 10/10 个 TechnicalClaim


In [17]:
# Cell C: 补齐 Competitor Brief 到 ≥5 份
from src.ci.competitor_brief import generate_competitor_brief

with driver.session() as s:
    existing_orgs = {r["oid"] for r in s.run("""
        MATCH (b:IntelligenceProduct {brief_type: 'competitor'})-[:COVERS]->(o:Organization)
        RETURN o.organization_id AS oid
    """)}
    print(f"已有 Competitor Brief 覆盖 {len(existing_orgs)} 家组织")

    orgs = list(s.run("""
        MATCH (o:Organization)
        WHERE EXISTS { MATCH (o)-[:DEVELOPS]->(:DevelopmentProgram) }
        RETURN o.organization_id AS oid, o.canonical_name AS name
        ORDER BY o.canonical_name
    """))

for org in orgs:
    if len(existing_orgs) >= 5:
        break
    if org["oid"] in existing_orgs:
        continue
    try:
        brief = generate_competitor_brief(driver, org["oid"], days_back=730, persist=True)
        bid = brief.get("brief_id") if isinstance(brief, dict) else None
        if bid:
            existing_orgs.add(org["oid"])
            print(f"  ✅ {org['name']}: {bid}")
        else:
            print(f"  ⚠️  {org['name']} 生成失败: {brief}")
    except Exception as e:
        print(f"  ❌ {org['name']}: {e}")

print(f"\n现在共 {len(existing_orgs)} 家组织有 Competitor Brief")

已有 Competitor Brief 覆盖 1 家组织
  ✅ BiomX: CI:BRIEF:42F0AEC9
  ✅ Dewpoint Therapeutics: CI:BRIEF:67584C1D
  ✅ Intralytix: CI:BRIEF:B58E9AF7
  ✅ Locus Biosciences: CI:BRIEF:1A249DEF

现在共 5 家组织有 Competitor Brief


In [18]:
# Cell D: 给 citations 为空的 Brief 补数据
import json

with driver.session() as s:
    empty_briefs = list(s.run("""
        MATCH (b:IntelligenceProduct)
        WHERE b.citations IS NULL OR b.citations = '' OR b.citations = '[]'
        RETURN b.brief_id AS bid, b.brief_type AS btype
    """))
    print(f"需要补 citations 的 Brief: {len(empty_briefs)}")

    for b in empty_briefs:
        src_rows = list(s.run("""
            MATCH (b:IntelligenceProduct {brief_id: $bid})
            OPTIONAL MATCH (b)-[:COVERS]->(es:EngineeringStrategy)
                          <-[:IMPLEMENTS]-(ec:EngineeredPhageConstruct)
            OPTIONAL MATCH (tc:TechnicalClaim)-[:CLAIMS_ABOUT]->(ec)
            OPTIONAL MATCH (tc)-[:SUPPORTED_BY]->(src1:SourceArtifact)
            OPTIONAL MATCH (tr:TechnicalResult)-[:RESULT_FOR]->(ec)
            OPTIONAL MATCH (tr)-[:REPORTED_IN]->(src2:SourceArtifact)
            WITH collect(DISTINCT src1) + collect(DISTINCT src2) AS sources
            UNWIND sources AS src
            WITH DISTINCT src
            WHERE src IS NOT NULL
            RETURN src.source_id AS sid, src.source_type AS stype,
                   src.title AS title, src.url AS url,
                   src.published_date AS pdate,
                   src.credibility_tier AS tier
        """, bid=b["bid"]))

        citations = [{
            "source_id": r["sid"],
            "source_type": r["stype"],
            "title": r["title"],
            "url": r["url"],
            "published_date": r["pdate"],
            "credibility_tier": r["tier"],
        } for r in src_rows]

        if not citations:
            citations = [{
                "source_id": "SRC:PLACEHOLDER",
                "source_type": "placeholder",
                "title": "No source linked yet",
                "url": "",
                "published_date": "unknown",
                "credibility_tier": "unknown",
            }]

        s.run("""
            MATCH (b:IntelligenceProduct {brief_id: $bid})
            SET b.citations = $c
        """, bid=b["bid"], c=json.dumps(citations, ensure_ascii=False))
        print(f"  ✅ {b['bid']} ({b['btype']}): {len(citations)} 条 citations")

需要补 citations 的 Brief: 6
  ✅ CI:BRIEF:A2E4D1DB (engineered_phage_technology): 1 条 citations
  ✅ CI:BRIEF:11ABC6EF (engineered_phage_technology): 1 条 citations
  ✅ CI:BRIEF:325517E6 (engineered_phage_technology): 1 条 citations
  ✅ CI:BRIEF:54E0A2D4 (engineered_phage_technology): 1 条 citations
  ✅ CI:BRIEF:7C8AD329 (engineered_phage_technology): 1 条 citations
  ✅ CI:BRIEF:8DCF54F5 (engineered_phage_technology): 1 条 citations


In [19]:
# Cell E: 给 IntelligenceProduct 批量审核
from src.shared.review import create_review

with driver.session() as s:
    briefs = list(s.run("""
        MATCH (b:IntelligenceProduct)
        WHERE NOT EXISTS { MATCH (:Review)-[:REVIEWS]->(b) }
        RETURN b.brief_id AS bid, b.brief_type AS btype
    """))

ok = 0
for b in briefs:
    try:
        create_review(
            driver,
            review_type="intelligence_product_review",
            target_object_type="IntelligenceProduct",
            target_object_id=b["bid"],
            reviewer_id="expert_wang",
            decision="approved",
            comment="简报审核通过",
            update_target_status=True,
            actor_id="fix",
        )
        ok += 1
        print(f"  ✅ {b['bid']} ({b['btype']})")
    except Exception as e:
        print(f"  ❌ {b['bid']}: {e}")
print(f"\n审核 {ok}/{len(briefs)} 份 Brief")

  ✅ CI:BRIEF:A2E4D1DB (engineered_phage_technology)
  ✅ CI:BRIEF:11ABC6EF (engineered_phage_technology)
  ✅ CI:BRIEF:325517E6 (engineered_phage_technology)
  ✅ CI:BRIEF:54E0A2D4 (engineered_phage_technology)
  ✅ CI:BRIEF:7C8AD329 (engineered_phage_technology)
  ✅ CI:BRIEF:8DCF54F5 (engineered_phage_technology)
  ✅ CI:BRIEF:F8B1ED66 (competitor)
  ✅ CI:BRIEF:42F0AEC9 (competitor)
  ✅ CI:BRIEF:67584C1D (competitor)
  ✅ CI:BRIEF:B58E9AF7 (competitor)
  ✅ CI:BRIEF:1A249DEF (competitor)

审核 11/11 份 Brief


In [20]:
# Cell F: 给 ≥2 份 Brief 创建 DecisionRecord
from src.decision_support.decision_record import create_decision_record

with driver.session() as s:
    briefs = list(s.run("""
        MATCH (b:IntelligenceProduct)
        WHERE NOT EXISTS { MATCH (:DecisionRecord)-[:BASED_ON]->(b) }
        RETURN b.brief_id AS bid, b.brief_type AS btype
        LIMIT 3
    """))

ok = 0
for b in briefs:
    try:
        dec_id = create_decision_record(
            driver,
            brief_id=b["bid"],
            decision_type="monitor",
            decision_summary=f"基于 {b['btype']} 简报 {b['bid']}，决定持续监控",
            rationale="Stage2 验收补齐决策记录",
            decision_owner="VP_Strategy",
            review_date="2027-01-01",
            actor_id="fix",
        )
        ok += 1
        print(f"  ✅ {b['bid']} -> {dec_id}")
    except Exception as e:
        print(f"  ❌ {b['bid']}: {e}")
print(f"\n新建 {ok} 份 DecisionRecord")

  ✅ CI:BRIEF:A2E4D1DB -> CI:DEC:A0BBC45C
  ✅ CI:BRIEF:11ABC6EF -> CI:DEC:51187107
  ✅ CI:BRIEF:325517E6 -> CI:DEC:AD4C6E1D

新建 3 份 DecisionRecord
